# Study 912 — Gold + Trend — the teardown

The excess-of-cash Sharpe race, the Newey-West return-difference *t*, the bootstrap Sharpe CIs, the random control, the two-era cut, the cost sweep, the IAU cross-check, and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `3ea3fb6649e8`).

In [1]:
R = {'start': '2007-05-30', 'end': '2026-06-30', 'n_days': 4802, 'fp': '3ea3fb6649e8', 'in_frac': 0.63, 'n_switches': 150, 'bh_sharpe': 0.52, 'bh_cagr': 8.08, 'bh_vol': 18.11, 'bh_dd': -45.56, 'bh_t': 2.37, 'tr_sharpe': 0.332, 'tr_cagr': 3.78, 'tr_vol': 14.23, 'tr_dd': -35.3, 'tr_t': 1.52, 'rnd_sharpe': 0.142, 'rnd_cagr': 1.02, 'rnd_dd': -55.33, 'adv': -0.188, 't_diff': -1.83, 'adv_vs_rnd': 0.19, 't_vs_rnd': 0.96, 'dd_impr': 10.25, 'ci_bh_lo': 0.093, 'ci_bh_hi': 0.943, 'ci_bh_neg': 0.9, 'ci_tr_lo': -0.105, 'ci_tr_hi': 0.753, 'ci_tr_neg': 5.9, 'era_e_n': 2164, 'era_e_bh': 0.34, 'era_e_tr': 0.06, 'era_e_adv': -0.28, 'era_e_t': -1.17, 'era_e_dd_bh': -45.6, 'era_e_dd_tr': -35.3, 'era_l_n': 2636, 'era_l_bh': 0.7, 'era_l_tr': 0.5, 'era_l_adv': -0.2, 'era_l_t': -1.7, 'era_l_dd_bh': -26.2, 'era_l_dd_tr': -30.2, 'cost0_adv': -0.16, 'cost0_t': -1.68, 'cost25_adv': -0.299, 'cost25_t': -2.39, 'iau_adv': -0.183, 'iau_t': -1.82, 'syn_pl_ddimpr': 23.4, 'syn_pl_adv': 0.14, 'syn_nl_mean': -0.05, 'syn_nl_sd': 0.13, 'syn_nl_fire': 0}

## The headline — excess-of-cash Sharpe race

GLD vs BIL, 200-day trend overlay vs buy-and-hold gold vs a 63%-matched random control. All excess-of-cash; drawdowns are absolute (lived).

In [2]:
print(f"buy-and-hold gold : exSharpe {R['bh_sharpe']:+.3f}  CAGR {R['bh_cagr']:+.2f}%  "
      f"vol {R['bh_vol']:.1f}%  DD {R['bh_dd']:.1f}%  HAC t {R['bh_t']:+.2f}")
print(f"200-day trend     : exSharpe {R['tr_sharpe']:+.3f}  CAGR {R['tr_cagr']:+.2f}%  "
      f"vol {R['tr_vol']:.1f}%  DD {R['tr_dd']:.1f}%  HAC t {R['tr_t']:+.2f}")
print(f"random control    : exSharpe {R['rnd_sharpe']:+.3f}  CAGR {R['rnd_cagr']:+.2f}%  "
      f"DD {R['rnd_dd']:.1f}%")
print(f"\nadvantage (trend - hold): {R['adv']:+.3f}  HAC t on return diff = {R['t_diff']:+.2f}")
print(f"advantage vs random     : {R['adv_vs_rnd']:+.3f}  (t = {R['t_vs_rnd']:+.2f})  "
      f"-> genuine timing, but it still loses to holding gold")
print(f"in-gold fraction {R['in_frac']:.1%}  |  {R['n_switches']} switches (~8/yr)")

buy-and-hold gold : exSharpe +0.520  CAGR +8.08%  vol 18.1%  DD -45.6%  HAC t +2.37
200-day trend     : exSharpe +0.332  CAGR +3.78%  vol 14.2%  DD -35.3%  HAC t +1.52
random control    : exSharpe +0.142  CAGR +1.02%  DD -55.3%

advantage (trend - hold): -0.188  HAC t on return diff = -1.83
advantage vs random     : +0.190  (t = +0.96)  -> genuine timing, but it still loses to holding gold
in-gold fraction 63.0%  |  150 switches (~8/yr)


## Bootstrap Sharpe CIs (excess-of-cash, 2,000 draws, 21-day blocks)

The overlay's Sharpe CI includes zero and sits *below* buy-and-hold's.

In [3]:
print(f"buy-and-hold: Sharpe {R['bh_sharpe']:+.3f}  95% CI [{R['ci_bh_lo']:+.3f}, {R['ci_bh_hi']:+.3f}]  share<0 {R['ci_bh_neg']:.1f}%")
print(f"trend       : Sharpe {R['tr_sharpe']:+.3f}  95% CI [{R['ci_tr_lo']:+.3f}, {R['ci_tr_hi']:+.3f}]  share<0 {R['ci_tr_neg']:.1f}%")

buy-and-hold: Sharpe +0.520  95% CI [+0.093, +0.943]  share<0 0.9%
trend       : Sharpe +0.332  95% CI [-0.105, +0.753]  share<0 5.9%


## Robustness — two eras (split 2016-01-01)

Advantage negative in **both** halves; the drawdown cushion is early-era only.

In [4]:
print(f"2007-2015 (n={R['era_e_n']}): BH {R['era_e_bh']:+.2f} / trend {R['era_e_tr']:+.2f}  "
      f"adv {R['era_e_adv']:+.2f} (t={R['era_e_t']:+.2f})  DD {R['era_e_dd_bh']:.1f}%->{R['era_e_dd_tr']:.1f}%")
print(f"2016-2026 (n={R['era_l_n']}): BH {R['era_l_bh']:+.2f} / trend {R['era_l_tr']:+.2f}  "
      f"adv {R['era_l_adv']:+.2f} (t={R['era_l_t']:+.2f})  DD {R['era_l_dd_bh']:.1f}%->{R['era_l_dd_tr']:.1f}%  <- deeper!")

2007-2015 (n=2164): BH +0.34 / trend +0.06  adv -0.28 (t=-1.17)  DD -45.6%->-35.3%
2016-2026 (n=2636): BH +0.70 / trend +0.50  adv -0.20 (t=-1.70)  DD -26.2%->-30.2%  <- deeper!


## Cost sweep + IAU cross-check

The edge is negative *before* any cost, and identical on a second gold ETF.

In [5]:
print(f"gross (0 bps): adv {R['cost0_adv']:+.3f} (t={R['cost0_t']:+.2f})")
print(f"25 bps      : adv {R['cost25_adv']:+.3f} (t={R['cost25_t']:+.2f})  -> only more negative")
print(f"IAU cross-check: adv {R['iau_adv']:+.3f} (t={R['iau_t']:+.2f})  -> not a GLD idiosyncrasy")

gross (0 bps): adv -0.160 (t=-1.68)
25 bps      : adv -0.299 (t=-2.39)  -> only more negative
IAU cross-check: adv -0.183 (t=-1.82)  -> not a GLD idiosyncrasy


## Live synthetic control — the machinery is unbiased

Planted dead-decade: the overlay MUST help. Flat-vol null: it must do nothing. This proves the real-tape miss is a property of gold, not the harness.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from gold_trend import data, strategy as st
pl = st.synthetic_detect(data.synthetic_daily(signal_strength=1.0, seed=912)[0])
print(f"planted: DD improvement {pl['dd_improvement']*100:+.1f} pp, excess-Sharpe adv {pl['excess_sharpe_adv']:+.2f}")
nl = np.array([st.synthetic_detect(data.synthetic_daily(signal_strength=0.0, seed=912+s)[0])['excess_sharpe_adv'] for s in range(8)])
print(f"null x8: excess-Sharpe adv mean {nl.mean():+.2f} (sd {nl.std(ddof=1):.2f}), |adv|>=0.4 in {(abs(nl)>=0.4).sum()}/8")

planted: DD improvement +23.4 pp, excess-Sharpe adv +0.14


null x8: excess-Sharpe adv mean -0.05 (sd 0.13), |adv|>=0.4 in 0/8


## Verdict

- **Signal — None.** The excess-of-cash Sharpe advantage is **-0.188** with the wrong sign (HAC *t* = -1.83); the overlay's bootstrap Sharpe CI [-0.105, +0.753] includes zero and sits below buy-and-hold's [+0.093, +0.943]; the advantage is negative in both eras (-0.28 / -0.20); the drawdown cushion (+10 pp overall) is early-era only and *reverses* post-2016. The rule beats a random control (real timing) but still loses to holding gold — the claimed better-Sharpe diversifier is absent. The synthetic control fires cleanly on a planted regime (adv +0.14) and is silent on the null (mean -0.05, 0/8), so the miss is real.
- **Tradability — Mirage.** Negative gross, more negative net (-0.299 at 25 bps). The overlay pays ~4.3 pp/yr of CAGR for a worst-loss cut that has vanished in the recent decade. Not bankable.